# Exploración de datos (EDA): churn de clientes de telecomunicaciones

Fase *Data Understanding* de CRISP-DM. El objetivo es conocer los datos antes de
limpiarlos y modelar: estructura, calidad (faltantes, duplicados, outliers),
la variable objetivo y qué variables se relacionan con el churn.

Las decisiones que salgan de aquí se implementarán en los pipelines de Kedro.

**Pregunta de negocio:** ¿qué clientes tienen mayor probabilidad de cancelar su
servicio, para que el equipo de retención los contacte primero?
(ver `docs/problema_de_negocio.md`)

## 1. Configuración y carga de datos

Los datos se cargan desde el **Data Catalog de Kedro** (`conf/base/catalog.yml`),
no con rutas escritas a mano. Así el notebook usa exactamente las mismas fuentes
que los pipelines.

Para ejecutarlo: abrir el notebook con el entorno del proyecto activo
(`jupyter lab` o VS Code) desde la carpeta `notebooks/` o la raíz del proyecto.

In [1]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from kedro.config import OmegaConfigLoader
from kedro.io import DataCatalog

# Aviso interno de Kedro/OmegaConf que no afecta el análisis
warnings.filterwarnings("ignore", message=".*register_new_resolver.*")

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)

In [2]:
# Trabajar desde la raíz del proyecto para que las rutas del catálogo funcionen
raiz = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(raiz)

# Leer el Data Catalog de Kedro (conf/base/catalog.yml)
config = OmegaConfigLoader(conf_source="conf", base_env="base", default_run_env="local")
catalog = DataCatalog.from_config(config["catalog"])
catalog.keys()

['raw_demographics',
 'raw_location',
 'raw_population',
 'raw_services',
 'raw_status',
 'raw_telco_churn',
 'raw_customer_churn']

In [3]:
demographics = catalog.load("raw_demographics")
location = catalog.load("raw_location")
population = catalog.load("raw_population")
services = catalog.load("raw_services")
status = catalog.load("raw_status")

tablas = {
    "demographics": demographics,
    "location": location,
    "population": population,
    "services": services,
    "status": status,
}
pd.DataFrame(
    {
        nombre: {"filas": df.shape[0], "columnas": df.shape[1]}
        for nombre, df in tablas.items()
    }
).T

,filas,columnas
demographics,7043,9
location,7043,10
population,1671,3
services,7043,31
status,7043,12
